# Tarea 2 · Parte 1 — Voces y documentos peruanos: del archivo escaneado al texto buscable

**Pregunta guía:** ¿Podemos convertir documentos escaneados y entrevistas en audio en un archivo de texto estructurado y buscable?

| Dato | Valor |
|---|---|
| Fecha de extracción | _(completar con la salida de la celda de versiones)_ |
| Entorno | `uv sync --group docai --group audio` (repositorio de la clase) |
| PaddleOCR | _(versión impresa abajo)_ — PP-OCR en español, `lang="es"` |
| ASR | faster-whisper large-v3-turbo (int8, CPU, VAD Silero activado) vía `voz.py` de la Sesión 13 |
| Diarización | huellas WeSpeaker ResNet34 (pyannote) + clustering jerárquico (`voz.diarizar`) |

**Fuentes y licencias** (detalle por archivo en `datos/fuentes.csv`):

- **Documentos:** Wikimedia Commons (dominio público, CC0, CC BY, CC BY-SA; autor y licencia por archivo) y capturas propias sin datos personales (CC BY 4.0).
- **Audio:** entrevistas de Wikimedia Commons (CC BY-SA 4.0), noticiarios de la Biblioteca Nacional del Perú en Commons (dominio público) y frases leídas de OpenSLR 73 *Crowdsourced high-quality Peruvian Spanish speech data set* (Google, CC BY-SA 4.0).

Los archivos originales quedan en `datos/raw/` (fuera de Git) y se recuperan con `uv run python descargas.py bajar`.

In [ ]:
import os, json, time, warnings
from datetime import date
from pathlib import Path

os.environ["PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK"] = "True"
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.express as px
import soundfile as sf

import paddleocr, faster_whisper
import herramientas as H
import voz

DATOS = Path("datos")
RAW_DOC, RAW_AUD = DATOS / "raw" / "documentos", DATOS / "raw" / "audio"
FECHA = date.today().isoformat()
pd.set_option("display.max_colwidth", 120)

print(f"Fecha de extracción: {FECHA}")
print(f"PaddleOCR {paddleocr.__version__} · faster-whisper {faster_whisper.__version__}")

## 1 · Corpus y fuentes

`descargas.py bajar` dejó los archivos en `datos/raw/` y su atribución en `datos/fuentes_crudas.csv`.
Primero se eliminan los duplicados exactos (mismo hash SHA-256).

In [ ]:
fuentes = pd.read_csv(DATOS / "fuentes_crudas.csv")
docs = fuentes[fuentes["modalidad"] == "documento"].reset_index(drop=True)
docs = H.marcar_duplicados(docs, "archivo")
descartes = docs.loc[docs["duplicado"], ["id", "archivo"]].assign(motivo="duplicado (mismo hash)")
docs = docs[~docs["duplicado"]].reset_index(drop=True)

print(f"Documentos únicos: {len(docs)} · duplicados descartados: {len(descartes)}")
docs.groupby("tipo").size().rename("n").to_frame()

## 2 · OCR con PaddleOCR

- **Idioma:** `lang="es"`.
- **Preprocesamiento:** corrección de rotación EXIF, RGB y lado máximo de 1600 px (copias en `datos/raw/documentos/prep/`).
- **Orientación por línea activada** (`use_textline_orientation=True`) para las fotos inclinadas; sin desdoblado de página.
- **Respaldo crudo:** cada resultado se guarda como JSON por línea (texto, confianza, caja) en `datos/raw/documentos/ocr_json/`, y las 3 líneas de menor confianza como recortes en `datos/raw/documentos/recortes/`.

In [ ]:
ocr = H.crear_ocr()
MODELO_OCR = f"PaddleOCR {paddleocr.__version__} (lang=es, textline_orientation=on, sin unwarping, lado máx. 1600 px)"
(RAW_DOC / "ocr_json").mkdir(parents=True, exist_ok=True)

filas = []
for d in docs.itertuples():
    prep = H.preprocesar(Path(d.archivo), RAW_DOC / "prep" / f"{d.id}.jpg")
    t0 = time.perf_counter()
    lineas = H.leer_ocr(ocr, prep)
    seg = time.perf_counter() - t0
    lineas.to_json(RAW_DOC / "ocr_json" / f"{d.id}.json", orient="records", force_ascii=False)
    H.recortes(lineas, prep, RAW_DOC / "recortes")
    filas.append({"id": d.id, "texto_ocr": H.unir_lineas(lineas), "n_lineas": len(lineas),
                  "confianza_media": lineas["confianza"].mean() if len(lineas) else 0.0, "segundos": round(seg, 2)})

ocr_res = docs.merge(pd.DataFrame(filas), on="id")
ocr_res[["id", "tipo", "n_lineas", "confianza_media", "segundos"]].round(3)

### Documentos ilegibles

Criterio fijado **antes** de mirar los resultados: un documento es ilegible si el OCR devuelve menos de 20 caracteres
o su confianza media es menor a 0.50. Si se descarta alguno, se reemplaza en `seleccion_documentos.csv`
por otro del **mismo tipo** y se vuelve a correr el notebook hasta tener 8 impresos, 8 fotos y 9 manuscritos.

In [ ]:
ilegible = (ocr_res["texto_ocr"].str.len() < 20) | (ocr_res["confianza_media"] < 0.50)
descartes = pd.concat([descartes, ocr_res.loc[ilegible, ["id", "archivo"]].assign(motivo="ilegible (<20 caracteres o confianza < 0.50)")])
ocr_res = ocr_res[~ilegible].reset_index(drop=True)

print(f"Ilegibles descartados: {ilegible.sum()}")
cuenta = ocr_res["tipo"].value_counts().reindex(["impreso", "foto", "manuscrito"]).fillna(0).astype(int)
minimos = pd.Series({"impreso": 8, "foto": 8, "manuscrito": 9})
pd.DataFrame({"tenemos": cuenta, "mínimo": minimos, "ok": cuenta >= minimos})

## 3 · ASR: clips, VAD, transcripción y diarización

1. Las entrevistas largas se cortan en **clips de ~2 minutos** siempre en una pausa (VAD Silero), para tener unidades comparables.
2. Las frases de OpenSLR 73 se usan como clips cortos de habla leída (un hablante).
3. Los clips con menos de 10 % de voz se descartan como «sin voz».
4. Transcripción con faster-whisper (`vad_filter=True`, que evita que Whisper alucine en silencios y música).
5. **Diarización** en entrevistas y noticiarios con `voz.diarizar` (número de hablantes estimado con umbral coseno 0.7).
6. Exportación de cada clip a **SRT** y **TXT** en `datos/raw/audio/salidas/`.

In [ ]:
CLIPS = RAW_AUD / "clips"; CLIPS.mkdir(parents=True, exist_ok=True)
fuentes_aud = fuentes[fuentes["modalidad"] == "audio"]

clips = []
for f in fuentes_aud.itertuples():
    audio = voz.cargar_audio(f.archivo)
    for k, (a, b) in enumerate(H.cortar_en_clips(audio)):
        ruta = CLIPS / f"{f.id}_{k:02d}.wav"
        sf.write(ruta, audio[int(a * voz.SR):int(b * voz.SR)], voz.SR)
        clips.append({"fuente_id": f.id, "tipo_audio": f.tipo, "archivo_audio": str(ruta),
                      "inicio_en_fuente": round(a, 1), "duracion_seg": round(b - a, 1)})

# frases leídas de OpenSLR 73 (descargadas por descargas.py en datos/raw/audio/es_peru/)
openslr = pd.read_csv(RAW_AUD / "es_peru" / "transcripciones.csv")
for r in openslr.itertuples():
    ruta = RAW_AUD / "es_peru" / r.archivo
    clips.append({"fuente_id": "openslr73", "tipo_audio": "habla leída", "archivo_audio": str(ruta),
                  "inicio_en_fuente": 0.0, "duracion_seg": round(sf.info(ruta).duration, 1)})

clips = pd.DataFrame(clips)
clips = H.marcar_duplicados(clips, "archivo_audio")
clips["prop_voz"] = [H.proporcion_voz(voz.cargar_audio(r)) for r in clips["archivo_audio"]]

sin_voz = clips["prop_voz"] < 0.10
descartes = pd.concat([descartes,
    clips.loc[clips["duplicado"], ["archivo_audio"]].assign(motivo="audio duplicado (mismo hash)"),
    clips.loc[sin_voz & ~clips["duplicado"], ["archivo_audio"]].assign(motivo="sin voz (<10 % de habla según VAD)")])
clips = clips[~clips["duplicado"] & ~sin_voz].reset_index(drop=True)
clips.insert(0, "id", [f"clip_{i:03d}" for i in range(1, len(clips) + 1)])

minutos = clips["duracion_seg"].sum() / 60
print(f"{len(clips)} clips · {minutos:.1f} minutos · descartados sin voz: {sin_voz.sum()}")
assert len(clips) >= 20 and minutos >= 30, "Faltan clips o minutos: agrega audios a seleccion_audio.csv"

In [ ]:
t = voz.Transcriptor("faster-whisper")
MODELO_ASR = "faster-whisper large-v3-turbo (int8, CPU, VAD on, beam 5)"
SALIDAS = RAW_AUD / "salidas"; SALIDAS.mkdir(exist_ok=True)
DIARIZAR = {"entrevista", "noticiario"}

filas = []
for c in clips.itertuples():
    audio = voz.cargar_audio(c.archivo_audio)
    t0 = time.perf_counter()
    segs = t.segmentos(audio, idioma="es")
    n_h = 1
    if c.tipo_audio in DIARIZAR and len(segs) > 1:
        segs = voz.diarizar(audio, segs)
        n_h = len({s.hablante for s in segs})
    seg_proc = time.perf_counter() - t0
    (SALIDAS / f"{c.id}.srt").write_text(voz.a_srt(segs), encoding="utf-8")
    (SALIDAS / f"{c.id}.txt").write_text(voz.a_texto(segs), encoding="utf-8")
    filas.append({"id": c.id, "texto_asr": H.texto_con_hablantes(segs) if n_h > 1 else " ".join(s.texto for s in segs),
                  "n_hablantes": n_h, "segundos_proceso": round(seg_proc, 1)})

asr_res = clips.merge(pd.DataFrame(filas), on="id")
asr_res["rtf"] = (asr_res["segundos_proceso"] / asr_res["duracion_seg"]).round(2)
print(f"Factor de tiempo real medio: {asr_res['rtf'].mean():.2f}")
print(f"Clips con 2+ hablantes: {(asr_res['n_hablantes'] >= 2).sum()}")
asr_res[["id", "tipo_audio", "duracion_seg", "n_hablantes", "rtf"]].head(10)

### Revisión manual contra alucinaciones

Se escuchan 5 clips al azar (semilla fija) y se comparan con su texto. Anotar aquí lo encontrado
(frases repetidas, texto inventado en la música de los noticiarios, etc.):

- _(completar después de escuchar)_

In [ ]:
from IPython.display import Audio, display
for r in asr_res.sample(5, random_state=7).itertuples():
    print(f"── {r.id} · {r.tipo_audio} · {r.duracion_seg:.0f} s\n{r.texto_asr[:400]}")
    display(Audio(r.archivo_audio))

## 4 · Seudonimización

1. **Patrones automáticos:** DNI (8 dígitos), RUC (11), teléfonos móviles y correos → `[DNI]`, `[RUC]`, `[TELEFONO]`, `[CORREO]`.
2. **Nombres de persona:** el modelo NER `mrm8488/bert-spanish-cased-finetuned-ner` *sugiere* candidatos; la decisión es humana.
   - Personas privadas (entrevistadas en la calle, nombres mencionados de terceros) → seudónimo (`Persona_A`, …).
   - Figuras públicas o históricas en documentos públicos (Ricardo Palma, firmantes del Acta de 1821, autoridades en ejercicio en una resolución oficial) **se conservan**, porque no son datos personales privados.
3. Revisión final leyendo los textos: el procedimiento automático no sustituye la revisión humana.

In [ ]:
todos = pd.concat([ocr_res.set_index("id")["texto_ocr"], asr_res.set_index("id")["texto_asr"]])
candidatos = H.sugerir_personas(todos)
candidatos.groupby("nombre").agg(veces=("id", "size"), ids=("id", lambda s: ", ".join(sorted(set(s))))).sort_values("veces", ascending=False)

In [ ]:
# ← COMPLETAR tras leer la tabla anterior: solo personas privadas
REEMPLAZOS = {
    # "Nombre Apellido": "Persona_A",
}

conteo_total = pd.Series(dtype=int)
for tabla, col in [(ocr_res, "texto_ocr"), (asr_res, "texto_asr")]:
    nuevos = []
    for texto in tabla[col]:
        limpio, c = H.seudonimizar_texto(texto, REEMPLAZOS)
        nuevos.append(limpio)
        conteo_total = conteo_total.add(pd.Series(c, dtype=int), fill_value=0)
    tabla[col] = nuevos

print(f"Entidades reemplazadas: {int(conteo_total.sum())}")
conteo_total.astype(int).rename("reemplazos").to_frame()

## 5 · Resumen de descartes

In [ ]:
print(f"Total descartados: {len(descartes)}")
descartes.fillna("").reset_index(drop=True)

## 6 · Análisis exploratorio

Los títulos de los gráficos se calculan a partir de los datos, para que expresen la conclusión y no solo la variable.

In [ ]:
ocr_res["caracteres_ocr"] = ocr_res["texto_ocr"].str.len()
asr_res["palabras_asr"] = asr_res["texto_asr"].str.replace(r"\[HABLANTE_\d+\]", "", regex=True).str.split().str.len()

conf = ocr_res.groupby("tipo")["confianza_media"].mean()
peor = conf.idxmin()
fig1 = px.histogram(ocr_res, x="tipo", color="tipo", category_orders={"tipo": ["impreso", "foto", "manuscrito"]},
                    title=f"El corpus está balanceado entre los tres tipos; los {peor}s tienen la menor confianza OCR ({conf[peor]:.2f})",
                    labels={"tipo": "Tipo de documento", "count": "Documentos"})
fig1.show()

In [ ]:
por_fuente = asr_res.groupby("tipo_audio")["duracion_seg"].sum().div(60).sort_values(ascending=False)
fig2 = px.bar(asr_res.sort_values("duracion_seg", ascending=False), x="id", y="duracion_seg", color="tipo_audio",
              title=f"Las entrevistas aportan {por_fuente.get('entrevista', 0) / por_fuente.sum():.0%} de los "
                    f"{por_fuente.sum():.0f} minutos; las frases de OpenSLR son muchas pero cortas",
              labels={"duracion_seg": "Duración (s)", "id": "Clip", "tipo_audio": "Tipo de audio"})
fig2.update_xaxes(showticklabels=False)
fig2.show()

In [ ]:
largos = pd.concat([
    ocr_res.assign(modalidad="OCR (caracteres)", longitud=ocr_res["caracteres_ocr"])[["modalidad", "longitud"]],
    asr_res.assign(modalidad="ASR (palabras)", longitud=asr_res["palabras_asr"])[["modalidad", "longitud"]],
])
med = largos.groupby("modalidad")["longitud"].median()
fig3 = px.histogram(largos, x="longitud", facet_col="modalidad", nbins=30, facet_col_spacing=0.08,
                    title=f"Las longitudes son muy desiguales: mediana de {med['OCR (caracteres)']:.0f} caracteres por documento "
                          f"y {med['ASR (palabras)']:.0f} palabras por clip",
                    labels={"longitud": "Longitud del texto extraído"})
fig3.update_xaxes(matches=None)
fig3.show()

## 7 · Validación: extremos de longitud

In [ ]:
cols = ["id", "tipo", "caracteres_ocr", "texto_ocr"]
print("Documentos con OCR más largo"); display(ocr_res.nlargest(3, "caracteres_ocr")[cols])
print("Documentos con OCR más corto"); display(ocr_res.nsmallest(3, "caracteres_ocr")[cols])
cols = ["id", "tipo_audio", "duracion_seg", "palabras_asr", "texto_asr"]
print("Clips más largos"); display(asr_res.nlargest(3, "duracion_seg")[cols])
print("Clips más cortos"); display(asr_res.nsmallest(3, "duracion_seg")[cols])

**Coherencia con los histogramas:** _(completar: ¿los documentos más largos son impresos de página completa y los más cortos fotos de carteles? ¿Las palabras de cada clip son proporcionales a su duración, unas 2 a 3 palabras por segundo?)_

## 8 · Exportación

In [ ]:
documentos = ocr_res.assign(modelo_ocr=MODELO_OCR, texto_referencia="")[
    ["id", "archivo", "tipo", "fuente", "texto_ocr", "modelo_ocr", "texto_referencia"]
    if "fuente" in ocr_res else ["id", "archivo", "tipo", "texto_ocr", "modelo_ocr", "texto_referencia"]]
if "fuente" not in documentos:
    documentos.insert(3, "fuente", ocr_res["fuente_url"].fillna("captura propia") + " (" + ocr_res["licencia"] + ")")

transcripciones = asr_res.assign(idioma="es (español peruano según la fuente)", modelo_asr=MODELO_ASR, texto_referencia="")[
    ["id", "archivo_audio", "duracion_seg", "idioma", "texto_asr", "modelo_asr", "n_hablantes", "texto_referencia",
     "fuente_id", "tipo_audio", "rtf"]]

openslr_fuente = pd.DataFrame([{"id": "openslr73", "modalidad": "audio", "fuente_url": "https://www.openslr.org/73/",
                                "licencia": "CC BY-SA 4.0", "autor": "Google (Guevara-Rukoz et al., 2020)",
                                "fecha_extraccion": FECHA, "acceso": "descargas.py bajar (datos.py de la Sesión 13, HTTP Range)"}])
fuentes_csv = pd.concat([fuentes, openslr_fuente])[["id", "modalidad", "fuente_url", "licencia", "acceso", "autor", "fecha_extraccion"]]

documentos.to_csv(DATOS / "documentos.csv", index=False)
transcripciones.to_csv(DATOS / "transcripciones.csv", index=False)
fuentes_csv.to_csv(DATOS / "fuentes.csv", index=False)

for nombre, df in [("documentos", documentos), ("transcripciones", transcripciones), ("fuentes", fuentes_csv)]:
    print(nombre, df.shape)
display(documentos.head())
display(transcripciones.head())